# 01. Importing libraries and data

In [1]:
# Importing necessary libraries
import pandas as pd
import numpy as np
import os

# Creating path string for project folder
path = r"/Users/a/Documents/Instacart Basket Analysis/"

# Importing orders.csv (same as 4.2)
orders_path = os.path.join(path, "02 Data", "Original Data", "orders.csv")
vars_list = ["order_id", "user_id", "order_number", "order_dow",
             "order_hour_of_day", "days_since_prior_order"]
df_ords = pd.read_csv(orders_path, usecols=vars_list)

# Importing products.csv (same as 4.2)
prods_path = os.path.join(path, "02 Data", "Original Data", "products.csv")
df_prods = pd.read_csv(prods_path)

# Displaying first few rows to confirm import
df_ords.head()

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,1,2,8,NaN
1,2398795,1,2,3,7,15.0
2,473747,1,3,3,12,21.0
3,2254736,1,4,4,7,29.0
4,431534,1,5,4,15,28.0


In [4]:
# 02. Data wrangling – converting identifier to suitable format

# Converting order_id from numeric to string (object) type
df_ords["order_id"] = df_ords["order_id"].astype("str")

# Quick check of the new data type
df_ords["order_id"].dtype

dtype('O')

In [6]:
# 03. Renaming an unintuitive variable (order_dow)

df_ords = df_ords.rename(columns={"order_dow": "order_day_of_week"})

# Confirm rename worked
df_ords.head()

,order_id,user_id,order_number,order_day_of_week,order_hour_of_day,days_since_prior_order
0,2539329,1,1,2,8,NaN
1,2398795,1,2,3,7,15.0
2,473747,1,3,3,12,21.0
3,2254736,1,4,4,7,29.0
4,431534,1,5,4,15,28.0


In [8]:
# 04. Busiest hour for placing orders

df_ords["order_hour_of_day"].value_counts().sort_index()

order_hour_of_day
0      22758
1      12398
2       7539
3       5474
4       5527
5       9569
6      30529
7      91868
8     178201
9     257812
10    288418
11    284728
12    272841
13    277999
14    283042
15    283639
16    272553
17    228795
18    182912
19    140569
20    104292
21     78109
22     61468
23     40043
Name: count, dtype: int64

In [44]:
df_ords["order_hour_of_day"].value_counts().idxmax()

10

The busiest order hour is 10:00, with the highest number of orders.

### 05. Meaning of department_id = 4
According to the data dictionary, a department_id value of 4 corresponds to **Produce**.

In [16]:
df_prods[df_prods["department_id"] == 4].head()

,product_id,product_name,aisle_id,department_id,prices
30,31,White Pearl Onions,123,4,7.5
42,43,Organic Clementines,123,4,11.5
44,45,European Cucumber,83,4,14.3
65,66,European Style Spring Mix,123,4,11.7
88,89,Yogurt Fruit Dip Sliced Apples,123,4,12.6


06. Breakfast items subset

In [18]:
# 06. Subsetting breakfast products

# Create a mask for breakfast department (department_id = 14)
breakfast_mask = df_prods["department_id"] == 14

# Create a new dataframe with only breakfast items
df_breakfast = df_prods[breakfast_mask]

# Quick checks
df_breakfast.head()
df_breakfast.shape

(1116, 5)

07. Dinner party items subset

In [20]:
# 07. Subset for dinner party departments

# Department IDs for alcohol, deli, beverages, meat/seafood
dinner_depts = [5, 7, 12, 20]

# Create subset
df_dinner = df_prods[df_prods["department_id"].isin(dinner_depts)]

# Quick checks
df_dinner.head()
df_dinner.shape

(7650, 5)

The dinner-party subset df_dinner contains 7,650 rows.

08. Extracting information for user_id = 1

In [22]:
# 08. Extract all records for user_id = 1
df_user1 = df_ords[df_ords["user_id"] == 1]

# Quick checks
df_user1.head()
df_user1.shape

(11, 6)

09. Basic statistics for user_id = 1

In [24]:
# Summary statistics for user_id = 1
df_user1.describe()

,user_id,order_number,order_day_of_week,order_hour_of_day,days_since_prior_order
count,11.0,11.000000,11.000000,11.000000,10.000000
mean,1.0,6.000000,2.636364,10.090909,19.000000
std,0.0,3.316625,1.286291,3.477198,9.030811
min,1.0,1.000000,1.000000,7.000000,0.000000
25%,1.0,3.500000,1.500000,7.500000,14.250000
50%,1.0,6.000000,3.000000,8.000000,19.500000
75%,1.0,8.500000,4.000000,13.000000,26.250000
max,1.0,11.000000,4.000000,16.000000,30.000000


In [26]:
# Additional insights
df_user1["order_number"].max()   # How many orders they've placed
df_user1["order_hour_of_day"].mode()  # Most common hour for this user
df_user1["days_since_prior_order"].mean()  # Average reorder interval

19.0

10. Notebook organization check

All section headings are included, and comments accompany each code block for clarity.

In [32]:
# 12. Exporting wrangled orders data
ords_export_path = os.path.join(path, "02 Data", "Prepared Data", "orders_wrangled.csv")
df_ords.to_csv(ords_export_path, index=False)

In [34]:
# Importing departments data
dept_path = os.path.join(path, "02 Data", "Original Data", "departments.csv")
df_dep = pd.read_csv(dept_path)

df_dep.head()

,department_id,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,department,frozen,other,bakery,produce,alcohol,international,beverages,pets,dry goods pasta,...,meat seafood,pantry,breakfast,canned goods,dairy eggs,household,babies,snacks,deli,missing


In [36]:
# 13. Creating departments lookup table (transpose method)

# Transpose the dataframe
df_dep_t = df_dep.T

# Reset index so department_id becomes a column
df_dep_t.reset_index(inplace=True)

# Rename the columns to match assignment requirements
df_dep_t.columns = ["department_id", "department"]

# Save final version as df_dep_t_new
df_dep_t_new = df_dep_t

# Quick check
df_dep_t_new.head()

,department_id,department
0,department_id,department
1,1,frozen
2,2,other
3,3,bakery
4,4,produce


In [42]:
# 14. Exporting wrangled departments data
dep_export_path = os.path.join(path, "02 Data", "Prepared Data", "departments_wrangled.csv")
df_dep_t_new.to_csv(dep_export_path, index=False)